# Real Embedding Analysis - CheXpert Race Fairness (Lightweight)

This notebook demonstrates shortcut detection on **real CheXpert data** with demographic information.

**What you'll learn:**
- How to load and merge real CheXpert data with demographics  
- How to detect race-based shortcuts in embeddings
- How to generate a comprehensive fairness report

**Data:** Uses 2000 random samples from CheXpert training set for fast computation.

**Data sources:**
- `data/train.csv` - CheXpert chest X-ray metadata
- `data/CHEXPERT DEMO.xlsx` - Patient demographic information (race)

In [53]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shortcut_detect import ShortcutDetector
from shortcut_detect import generate_linear_shortcut

print("✅ Imports successful!")

✅ Imports successful!


In [54]:
# Setup paths - works from any directory
import os
import sys

# Get the notebook directory
notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()

# Find project root (go up until we find data/ folder)
current = notebook_dir
for _ in range(5):  # Max 5 levels up
    if os.path.exists(os.path.join(current, 'data', 'train.csv')):
        project_root = current
        break
    current = os.path.dirname(current)
else:
    # Fallback: assume we're in examples/02_real_data/
    project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..'))

print(f"Project root: {project_root}")

# Set paths
data_dir = os.path.join(project_root, 'data')
output_dir = os.path.join(project_root, 'examples', 'output_real_data')
csv_dir = os.path.join(output_dir, 'csv_exports')

# Create output directories
os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")
print("✅ Paths configured!")

Project root: /home/sebasmos/Desktop/Shortcut_Detect
Data directory: /home/sebasmos/Desktop/Shortcut_Detect/data
Output directory: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data
✅ Paths configured!


## 1. Load Real CheXpert Data with Demographics

We'll load the CheXpert training data and merge it with demographic information to get race labels.

In [55]:
# Load CheXpert training data
print("Loading CheXpert training data...")
train_csv_path = os.path.join(data_dir, 'train.csv')
data_df = pd.read_csv(train_csv_path)
print(f"✅ Loaded {len(data_df)} chest X-ray samples")
print(f"\nColumns: {list(data_df.columns[:10])}...")

Loading CheXpert training data...
✅ Loaded 223414 chest X-ray samples

Columns: ['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion']...


In [56]:
# Extract patient IDs from paths
print("\nExtracting patient IDs...")
path_split = data_df.Path.str.split("/", expand=True)
data_df["patient_id"] = path_split[2]
print(f"✅ Extracted {data_df['patient_id'].nunique()} unique patients")


Extracting patient IDs...
✅ Extracted 64540 unique patients


In [57]:
# Load demographic data
print("\nLoading demographic data...")
demo_xlsx_path = os.path.join(data_dir, 'CHEXPERT DEMO.xlsx')
demo_df = pd.read_excel(demo_xlsx_path, engine='openpyxl')
demo_df = demo_df.rename(columns={'PATIENT': 'patient_id'})
print(f"✅ Loaded demographics for {len(demo_df)} patients")
print(f"\nDemographic columns: {list(demo_df.columns)}")


Loading demographic data...
✅ Loaded demographics for 65401 patients

Demographic columns: ['patient_id', 'GENDER', 'AGE_AT_CXR', 'PRIMARY_RACE', 'ETHNICITY']


In [58]:
# Merge CheXpert data with demographics
print("\nMerging data...")
data_df = data_df.merge(demo_df, on="patient_id", how="left")
print(f"✅ Merged data: {len(data_df)} samples with demographics")


Merging data...
✅ Merged data: 223414 samples with demographics


## 2. Process Race Labels

Extract and encode race information for fairness analysis.

In [59]:
# Create race column from PRIMARY_RACE
data_df["race"] = None

# Map race categories
mask = (data_df.PRIMARY_RACE.str.contains("Black", na=False))
data_df.loc[mask, "race"] = "BLACK/AFRICAN AMERICAN"

mask = (data_df.PRIMARY_RACE.str.contains("White", na=False))
data_df.loc[mask, "race"] = "WHITE"

mask = (data_df.PRIMARY_RACE.str.contains("Asian", na=False))
data_df.loc[mask, "race"] = "ASIAN"

# Drop rows with unknown race
data_df = data_df[~data_df["race"].isna()].reset_index(drop=True)

# SAMPLE 2000 random samples for lightweight demo
print(f"Total samples with race: {len(data_df)}")
if len(data_df) > 2000:
    data_df = data_df.sample(n=2000, random_state=42).reset_index(drop=True)
    print(f"⚡ Sampled {len(data_df)} for faster computation")

print(f"\n✅ Using {len(data_df)} samples")
print(f"\nRace distribution:")
print(data_df['race'].value_counts())
print(f"\nRace percentages:")
print(data_df['race'].value_counts(normalize=True) * 100)

Total samples with race: 160724
⚡ Sampled 2000 for faster computation

✅ Using 2000 samples

Race distribution:
race
WHITE                     1531
ASIAN                      311
BLACK/AFRICAN AMERICAN     158
Name: count, dtype: int64

Race percentages:
race
WHITE                     76.55
ASIAN                     15.55
BLACK/AFRICAN AMERICAN     7.90
Name: proportion, dtype: float64


In [60]:
# Encode race as 0/1/2
label_to_idx = {"ASIAN": 0, "BLACK/AFRICAN AMERICAN": 1, "WHITE": 2}
data_df["race_idx"] = data_df["race"].map(label_to_idx)

print("Race encoding:")
for race, idx in label_to_idx.items():
    count = (data_df["race_idx"] == idx).sum()
    print(f"  {idx}: {race} ({count} samples)")

# Display sample
print("\nSample data with race labels:")
data_df[['Path', 'Sex', 'Age', 'race', 'race_idx']].head(10)

Race encoding:
  0: ASIAN (311 samples)
  1: BLACK/AFRICAN AMERICAN (158 samples)
  2: WHITE (1531 samples)

Sample data with race labels:


,Path,Sex,Age,race,race_idx
0,CheXpert-v1.0-small/train/patient64369/study1/...,Male,78,WHITE,2
1,CheXpert-v1.0-small/train/patient59605/study1/...,Male,66,WHITE,2
2,CheXpert-v1.0-small/train/patient29605/study7/...,Male,61,WHITE,2
3,CheXpert-v1.0-small/train/patient06733/study1/...,Male,74,WHITE,2
4,CheXpert-v1.0-small/train/patient25140/study1/...,Female,62,WHITE,2
5,CheXpert-v1.0-small/train/patient22703/study5/...,Male,37,WHITE,2
6,CheXpert-v1.0-small/train/patient28055/study2/...,Male,72,WHITE,2
7,CheXpert-v1.0-small/train/patient12526/study17...,Female,68,WHITE,2
8,CheXpert-v1.0-small/train/patient25125/study27...,Male,58,ASIAN,0
9,CheXpert-v1.0-small/train/patient07932/study12...,Female,41,WHITE,2


## 3. Generate Embeddings

**Note for production use:**
In a real scenario, you would extract embeddings from your trained model:

```python
# Example: Extract ResNet50 embeddings from images
import torch
from torchvision import models

model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])
model.eval()

# Extract features for each image
embeddings = []
for image_path in data_df['Path']:
    image = load_and_preprocess_image(image_path)
    embedding = model(image)
    embeddings.append(embedding)
```

For this demo, we generate synthetic 2048-dimensional embeddings that simulate ResNet50 features:

In [61]:
# Generate lightweight embeddings (512-dim instead of 2048 for speed)
n_samples = len(data_df)
embedding_dim = 512   # Lighter than ResNet50 (2048)
shortcut_dims = 10    # Simulate race-based shortcuts

print(f"\nGenerating lightweight embeddings for {n_samples} samples...")
print(f"Using {embedding_dim}-dim embeddings (4x smaller than ResNet50)")

embeddings, _ = generate_linear_shortcut(
    n_samples=n_samples,
    embedding_dim=embedding_dim,
    shortcut_dims=shortcut_dims,
    seed=42
)

# Use the REAL race labels from CheXpert data
race_labels = data_df["race_idx"].values

print(f"\n✅ Generated embeddings: {embeddings.shape}")
print(f"✅ Race labels: {race_labels.shape}")


Generating lightweight embeddings for 2000 samples...
Using 512-dim embeddings (4x smaller than ResNet50)

✅ Generated embeddings: (2000, 512)
✅ Race labels: (2000,)


## 4. Run Comprehensive Shortcut Detection

Detect race-based shortcuts using all three methods:
- **HBAC**: Hierarchical clustering analysis
- **Probe**: Linear classifier probe
- **Statistical**: Feature-wise statistical tests

In [62]:
# Create detector with all methods
detector = ShortcutDetector(
    methods=['hbac', 'probe', 'statistical'],
    seed=42
)

print("="*70)
print("RUNNING RACE-BASED SHORTCUT DETECTION")
print(f"Dataset: {n_samples} CheXpert samples")
print(f"Protected attribute: Race (Asian, Black/African American, White)")
print("="*70)

# Fit on embeddings with real race labels
detector.fit(embeddings, race_labels)

print("\n✅ Detection complete!")

RUNNING RACE-BASED SHORTCUT DETECTION
Dataset: 2000 CheXpert samples
Protected attribute: Race (Asian, Black/African American, White)
Running HBAC detection...
Running probe-based detection...
Running statistical tests...
✅ Detection complete!

✅ Detection complete!


In [63]:
# View comprehensive summary
print(detector.summary())

UNIFIED SHORTCUT DETECTION SUMMARY
Dataset: 2000 samples, 512 dimensions
Methods used: hbac, probe, statistical

----------------------------------------------------------------------
HBAC (Clustering-based Detection)
----------------------------------------------------------------------
Shortcut detected: NO
Confidence: low
Clusters found: 4

----------------------------------------------------------------------
Probe-based Detection
----------------------------------------------------------------------
Probe accuracy: 90.65%
⚠️  High probe accuracy suggests strong shortcut signal!

----------------------------------------------------------------------
Statistical Testing
----------------------------------------------------------------------
Comparisons performed: 6
Comparisons with significant features: 6
  [0_vs_1]: 33 significant features
  [0_vs_2]: 33 significant features
  [1_vs_2]: 31 significant features
  [0_vs_rest]: 31 significant features
  [1_vs_rest]: 32 significant feat

## 5. Generate Single Comprehensive Report

Generate ONE complete fairness report with:
- All detection results (HBAC, Probe, Statistical)
- All visualizations (PCA, t-SNE, heatmaps, dimension importance)
- Complete statistical analysis
- CSV exports for detailed analysis

In [64]:
# Generate ONE comprehensive report with everything
report_path = os.path.join(output_dir, 'chexpert_race_fairness_comprehensive_report.html')

detector.generate_report(
    output_path=report_path,
    format='html',
    include_visualizations=True,  # Include ALL visualizations
    export_csv=True,              # Export detailed CSV files
    csv_dir=csv_dir
)

# Close all matplotlib figures to free memory
plt.close('all')

print("\n" + "="*70)
print("✅ COMPREHENSIVE REPORT GENERATED")
print("="*70)
print(f"\n📊 Single HTML report: {report_path}")
print(f"📁 CSV exports: {csv_dir}/")
print(f"\nReport includes:")
print("  ✓ HBAC clustering analysis")
print("  ✓ Probe classification results")
print("  ✓ Statistical test outcomes")
print("  ✓ PCA visualization")
print("  ✓ t-SNE visualization")
print("  ✓ Dimension importance plot")
print("  ✓ Cluster purity heatmap")
print("  ✓ P-value heatmap")
print("  ✓ 7 CSV files with detailed metrics")
print("\n✅ Done! All figures closed.")

Generating visualizations...
  Generating PCA plot...
  Generating t-SNE plot...
  Generating dimension importance plot...
  Generating cluster purity plot...
  Generating p-value heatmap...
✅ HTML report saved to: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/chexpert_race_fairness_comprehensive_report.html
Exporting to CSV...
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/overall_summary.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/hbac_cluster_purities.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/hbac_dimension_importance.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/probe_predictions.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/probe_summary.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/statist

## 6. Optional: Generate PDF Report

Create a PDF version for sharing with clinical stakeholders.

In [ ]:
# Optional: Generate PDF version of the comprehensive report
# Requires weasyprint + system dependencies
# On macOS: brew install pango gdk-pixbuf libffi
# On Ubuntu: apt-get install libpango-1.0-0 libpangocairo-1.0-0

pdf_path = os.path.join(output_dir, 'chexpert_race_fairness_comprehensive_report.pdf')

try:
    detector.generate_report(
        output_path=pdf_path,
        format='pdf',
        include_visualizations=True
    )
    print(f"✅ PDF report generated: {pdf_path}")
except OSError as e:
    if "libgobject" in str(e) or "library" in str(e).lower():
        print("⚠️  PDF generation skipped - missing system dependencies")
        print("   To enable PDF export, install required libraries:")
        print("   • macOS: brew install pango gdk-pixbuf libffi")
        print("   • Ubuntu: apt-get install libpango-1.0-0 libpangocairo-1.0-0")
    else:
        raise

# Close all matplotlib figures
plt.close('all')
print("✅ Done! All figures closed.")

## 7. Save Data for Further Analysis

In [66]:
# Save embeddings and metadata for external analysis
embeddings_path = os.path.join(output_dir, 'chexpert_embeddings.npy')
metadata_path = os.path.join(output_dir, 'chexpert_metadata_with_race.csv')

np.save(embeddings_path, embeddings)
data_df[['Path', 'patient_id', 'Sex', 'Age', 'race', 'race_idx']].to_csv(
    metadata_path, 
    index=False
)

print("✅ Saved for external analysis:")
print(f"   - Embeddings: {embeddings_path}")
print(f"   - Metadata: {metadata_path}")

✅ Saved for external analysis:
   - Embeddings: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/chexpert_embeddings.npy
   - Metadata: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/chexpert_metadata_with_race.csv
